In [1]:
import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv("../data/tmdb_imdb_movies_cleaned.csv")

In [3]:
df.head()

,id,title,vote_average,vote_count,status,release_date,revenue,runtime,adult,backdrop_path,...,genres,production_companies,production_countries,spoken_languages,keywords,directors,writers,averageRating,numVotes,cast
0,2,Ariel,7.100,262,Released,1988-10-21,0,73,False,/dQL2wJZo05GDd21VgOacMeCuyZy.jpg,...,"Drama, Comedy, Romance",Villealfa Filmproductions,Finland,Finnish,"prison, underdog, helsinki, finland, factory w...",Aki Kaurismäki,Aki Kaurismäki,7.4,9872,"Turo Pajala, Susanna Haavisto, Matti Pellonpää..."
1,3,Shadows in Paradise,7.199,281,Released,1986-10-17,0,74,False,/l94l89eMmFKh7na2a1u5q67VgNx.jpg,...,"Drama, Comedy, Romance",Villealfa Filmproductions,Finland,"English, Finnish, Swedish","helsinki, finland, salesclerk, garbage",Aki Kaurismäki,Aki Kaurismäki,7.4,8729,"Matti Pellonpää, Kati Outinen, Sakari Kuosmane..."
2,5,Four Rooms,5.784,2436,Released,1995-12-09,4257354,98,False,/f2t4JbUvQIjUF5FstG1zZFAp02N.jpg,...,Comedy,"Miramax, A Band Apart",United States of America,English,"hotel, new year's eve, witch, bet, sperm, hote...","Allison Anders, Alexandre Rockwell, Robert Rod...","Allison Anders, Alexandre Rockwell, Robert Rod...",6.7,117182,"Tim Roth, Jennifer Beals, Antonio Banderas, Va..."
3,6,Judgment Night,6.533,302,Released,1993-10-15,12136938,109,False,/bGMqHn0H2UY6SPZ5Vch4YJM2jDO.jpg,...,"Action, Crime, Thriller","Largo Entertainment, JVC",United States of America,English,"drug dealer, chicago, illinois, escape, one ni...",Stephen Hopkins,"Lewis Colick, Jere Cunningham",6.6,21178,"Emilio Estevez, Cuba Gooding Jr., Denis Leary,..."
4,8,Life in Loops (A Megacities RMX),7.700,25,Released,2006-01-01,0,80,False,NaN,...,Documentary,inLoops,Austria,"English, Hindi, Japanese, Russian, Spanish",megacities,Timo Novotny,"Michael Glawogger, Timo Novotny",8.1,285,NaN


In [4]:
df.shape

(429354, 29)

In [5]:
df.columns.tolist()

['id',
 'title',
 'vote_average',
 'vote_count',
 'status',
 'release_date',
 'revenue',
 'runtime',
 'adult',
 'backdrop_path',
 'budget',
 'homepage',
 'tconst',
 'original_language',
 'original_title',
 'overview',
 'popularity',
 'poster_path',
 'tagline',
 'genres',
 'production_companies',
 'production_countries',
 'spoken_languages',
 'keywords',
 'directors',
 'writers',
 'averageRating',
 'numVotes',
 'cast']

In [6]:
columns = [
    "id",
    "title",
    "overview",
    "genres",
    "cast",
    "directors",
    "keywords",
    "release_date",
    "vote_average",
    "vote_count",
    "runtime"
]


df = df[columns]

In [7]:
df.columns.tolist()

['id',
 'title',
 'overview',
 'genres',
 'cast',
 'directors',
 'keywords',
 'release_date',
 'vote_average',
 'vote_count',
 'runtime']

In [9]:
df["release_date"] = pd.to_datetime(
    df["release_date"],
    errors="coerce"
)

df["year"] = df["release_date"].dt.year

In [10]:
df = df.drop_duplicates(subset=["title", "year"])

In [11]:
df = df[
    (df["year"] >= 1998) &
    (df["overview"].notna()) &
    (df["overview"].str.strip() != "") &
    (df["vote_count"] >= 100) &
    (df["vote_average"] >= 5.0)
]

In [13]:
df.shape

(12338, 12)

In [12]:
df.isnull().sum()

id                 0
title              0
overview           0
genres             2
cast              44
directors          7
keywords        1362
release_date       0
vote_average       0
vote_count         0
runtime            0
year               0
dtype: int64

In [18]:
fill_columns = [
    "genres",
    "cast",
    "directors",
    "keywords"
]


for col in fill_columns:
    df[col] = df[col].fillna("Unknown")

In [19]:
df.isnull().sum()

id              0
title           0
overview        0
genres          0
cast            0
directors       0
keywords        0
release_date    0
vote_average    0
vote_count      0
runtime         0
year            0
dtype: int64

In [20]:
df = df[
    df["vote_count"] >= 100
]

In [21]:
df = df[
    df["vote_average"] >= 5.0
]

In [23]:
text_columns = [
    "title",
    "overview",
    "genres",
    "cast",
    "directors",
    "keywords"
]


for col in text_columns:
    df[col] = (
        df[col]
        .astype(str)
        .str.strip()
    )

In [30]:
def convert_to_text(x):
    if isinstance(x, list):
        return ", ".join(x)
    if pd.isna(x):
        return ""
    return str(x)


cols = [
    "title",
    "overview",
    "genres",
    "directors",
    "cast",
    "keywords",
    "release_date",
    "vote_average",
    "runtime"
]

for col in cols:
    df[col] = df[col].apply(convert_to_text)

In [31]:
df.dtypes

id                int64
title            object
overview         object
genres           object
cast             object
directors        object
keywords         object
release_date     object
vote_average     object
vote_count        int64
runtime          object
year            float64
dtype: object

In [43]:
#create document for embedding
df["document"] = (
    "Title: " + df["title"] +
    "\nOverview: " + df["overview"] +
    "\nGenres: " + df["genres"] +
    "\nDirector: " + df["directors"] +
    "\nCast: " + df["cast"] +
    "\nKeywords: " + df["keywords"] +
    "\nRelease Date: " + df["release_date"] +
    "\nRating: " + df["vote_average"] +
    "\nRuntime: " + df["runtime"]
)

In [45]:
print(df["document"].iloc[0])


Title: Finding Nemo
Overview: Nemo, an adventurous young clownfish, is unexpectedly taken from his Great Barrier Reef home to a dentist's office aquarium. It's up to his worrisome father Marlin and a friendly but forgetful fish Dory to bring Nemo home -- meeting vegetarian sharks, surfer dude turtles, hypnotic jellyfish, hungry seagulls, and more along the way.
Genres: Animation, Family
Director: Andrew Stanton, Lee Unkrich
Cast: Albert Brooks, Ellen DeGeneres, Alexander Gould, Willem Dafoe, Geoffrey Rush, Brad Garrett, Allison Janney, Austin Pendleton, Stephen Root, Vicki Lewis
Keywords: sydney, australia, parent child relationship, anthropomorphism, harbor, underwater, shark, pelican, fish tank, great barrier reef, sea turtle, missing child, aftercreditsstinger, duringcreditsstinger, short term memory loss, clownfish, father son reunion, protective father
Release Date: 2003-05-30 00:00:00
Rating: 7.824
Runtime: 100


In [46]:
df.to_parquet(
    "../data/movie_dataset_clean.parquet",
    index=False
)

In [47]:
clean_df = pd.read_parquet(
    "../data/movie_dataset_clean.parquet"
)

clean_df.head()

,id,title,overview,genres,cast,directors,keywords,release_date,vote_average,vote_count,runtime,year,document
0,12,Finding Nemo,"Nemo, an adventurous young clownfish, is unexp...","Animation, Family","Albert Brooks, Ellen DeGeneres, Alexander Goul...","Andrew Stanton, Lee Unkrich","sydney, australia, parent child relationship, ...",2003-05-30 00:00:00,7.824,18061,100,2003.0,"Title: Finding Nemo\nOverview: Nemo, an advent..."
1,14,American Beauty,"Lester Burnham, a depressed suburban father in...",Drama,"Kevin Spacey, Annette Bening, Thora Birch, Wes...",Sam Mendes,"estate agent, adultery, coming out, first time...",1999-09-15 00:00:00,8.0,11260,122,1999.0,Title: American Beauty\nOverview: Lester Burnh...
2,16,Dancer in the Dark,"Selma, a Czech immigrant on the verge of blind...","Drama, Crime","Björk, Catherine Deneuve, David Morse, Peter S...",Lars von Trier,"factory worker, dying and death, individual, i...",2000-06-30 00:00:00,7.869,1618,141,2000.0,"Title: Dancer in the Dark\nOverview: Selma, a ..."
3,17,The Dark,"In an attempt to pull her family together, Adè...","Horror, Thriller, Mystery","Maria Bello, Sean Bean, Abigail Stone, Richard...",John Fawcett,"sea, wales, child abuse, shepherd, adolescence...",2005-09-28 00:00:00,5.755,247,87,2005.0,Title: The Dark\nOverview: In an attempt to pu...
4,20,My Life Without Me,A fatally ill mother with only two months to l...,"Drama, Romance","Sarah Polley, Amanda Plummer, Scott Speedman, ...",Isabel Coixet,"dying and death, daughter, farewell, night shi...",2003-03-07 00:00:00,5.941,421,106,2003.0,Title: My Life Without Me\nOverview: A fatally...
